In [ ]:
# NDVI Land Cover Classification - Advanced Pipeline
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data
df = pd.read_csv('train.csv')
y = df['class']
X = df.drop(columns=['class', 'ID'])

# 2. Apply Noise Filtering - Savitzky–Golay Smoothing
X_smooth = savgol_filter(X.values, window_length=5, polyorder=2, axis=1)
X = pd.DataFrame(X_smooth, columns=X.columns)

# 3. Impute Missing NDVI Values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# 4. Feature Engineering
X_feat = pd.DataFrame({
    'ndvi_mean': X_imputed.mean(axis=1),
    'ndvi_std': X_imputed.std(axis=1),
    'ndvi_max': X_imputed.max(axis=1),
    'ndvi_min': X_imputed.min(axis=1),
    'ndvi_range': X_imputed.max(axis=1) - X_imputed.min(axis=1),
    'ndvi_kurt': X_imputed.kurtosis(axis=1),
    'ndvi_skew': X_imputed.skew(axis=1),
    'ndvi_slope': np.polyfit(np.arange(X_imputed.shape[1]), X_imputed.values.T, 1)[0]
})

# 5. Adversarial Validation to Weight Clean-Like Samples Higher
X_train_adv, X_clean_sim, _, _ = train_test_split(X_feat, np.zeros(len(X_feat)), test_size=0.2, random_state=42)
y_adv = np.concatenate([np.zeros(len(X_train_adv)), np.ones(len(X_clean_sim))])
X_adv = pd.concat([X_train_adv, X_clean_sim])
adv_clf = RandomForestClassifier(n_estimators=50, random_state=42)
adv_clf.fit(X_adv, y_adv)
adv_prob = adv_clf.predict_proba(X_feat)[:, 1]
sample_weights = 1 / (1 + np.exp(5 * (adv_prob - 0.5)))  # Sigmoid smoothing

# 6. Pipeline & Hyperparameter Tuning
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000))
])
param_grid = {'clf__C': [0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid.fit(X_feat, y, clf__sample_weight=sample_weights)

print("Best Hyperparameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

# 7. Predict on Test Set
df_test = pd.read_csv('test.csv')
X_test_raw = df_test.drop(columns=['ID'])
X_test_smooth = savgol_filter(X_test_raw.values, window_length=5, polyorder=2, axis=1)
X_test_imputed = pd.DataFrame(imputer.transform(pd.DataFrame(X_test_smooth, columns=X_test_raw.columns)))

X_test_feat = pd.DataFrame({
    'ndvi_mean': X_test_imputed.mean(axis=1),
    'ndvi_std': X_test_imputed.std(axis=1),
    'ndvi_max': X_test_imputed.max(axis=1),
    'ndvi_min': X_test_imputed.min(axis=1),
    'ndvi_range': X_test_imputed.max(axis=1) - X_test_imputed.min(axis=1),
    'ndvi_kurt': X_test_imputed.kurtosis(axis=1),
    'ndvi_skew': X_test_imputed.skew(axis=1),
    'ndvi_slope': np.polyfit(np.arange(X_test_imputed.shape[1]), X_test_imputed.values.T, 1)[0]
})

predictions = grid.best_estimator_.predict(X_test_feat)

# 8. Save to Submission File
submission = pd.DataFrame({'ID': df_test['ID'], 'class': predictions})
submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")